In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [8]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    #text = chat(messages)
    
    return json.loads(text)

In [10]:
dataset = generate_dataset()

dataset

[{'task': "Parse an AWS S3 object key to extract the bucket name, folder path, and file name from a full S3 URI like 's3://my-bucket/path/to/file.txt'",
  'format': 'regex'},
 {'task': 'Create a Python function that takes an AWS EC2 instance ID and returns True if it matches the valid format (i-followed by 17 hexadecimal characters)',
  'format': 'python'},
 {'task': 'Generate a JSON configuration object for an AWS Lambda function that includes function name, runtime, handler, role ARN, timeout, and memory allocation',
  'format': 'json'}]

In [11]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [18]:

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explaination
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages,  stop_sequences=["```"])
    return output



In [19]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [20]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [21]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output, 
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
        }  

    

In [22]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

Average score: 8.333333333333334


In [55]:
print(json.dumps(result, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - s3://my-bucket.s3.us-west-2.amazonaws.com\n    - https://my-bucket.s3.us-west-2.amazonaws.com\n    - https://s3.us-west-2.amazonaws.com/my-bucket\n    - s3://my-bucket.s3.amazonaws.com (returns None for default region)\n    \n    Args:\n        url: S3 bucket URL string\n        \n    Returns:\n        AWS region string (e.g., 'us-west-2') or None if region not found\n        \n    Raises:\n        ValueError: If URL format is invalid\n    \"\"\"\n    if not url:\n        raise ValueError(\"URL cannot be empty\")\n    \n    # Pattern 1: Virtual-hosted style with region\n    # Format: bucket.s3.region.amazonaws.com or s3.region.amazonaws.com/bucket\n    patt